# D6 Transformer、生成、BLEU 与 ROUGE

**竞赛模块：D · 自然语言处理应用开发** · **建议学习约 40 分钟**

Transformer、GPT 和指标最容易混淆：模型生成文本是一回事，评价文本的质量是另一回事。先掌握评测目标，再学习调用真实模型。

> 零基础学习顺序：先读术语和说明 → 运行示例 → 理解代码 → 完成练习 → 概念自测。按 Shift+Enter 运行单元格。

## 一、先认识术语

验证 Transformer 前向传播，再对相同参考与候选序列计算标准 BLEU-2 和基于 LCS 的 ROUGE-L。
- **BLEU**：机器翻译常用的 n-gram 匹配指标
- **ROUGE-L**：利用最长公共子序列衡量摘要和参考文本的重合度

Transformer 使用注意力处理上下文；生成时需要指定停止条件。BLEU 评估机器翻译的 n-gram 精度，ROUGE 常用于摘要结果的词语重合度。

## 二、运行示例

逐格执行并查看输出。这里的代码使用本地/合成数据，遇到缺失输入会有提示。不要直接跳过基础代码。

In [ ]:
import torch
from torch import nn
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# 微型 Transformer 前向传播，示例是随机输入，没有进行模型训练
torch.manual_seed(42)
transformer = nn.Transformer(
    d_model=8, nhead=2, num_encoder_layers=1,
    num_decoder_layers=1, dim_feedforward=16,
    dropout=0.0, batch_first=True
)
transformer.eval()
with torch.no_grad():
    output = transformer(torch.randn(2, 3, 8), torch.randn(2, 2, 8))
print("Transformer 输出:", tuple(output.shape))

reference = ["人工", "智能", "发展"]
prediction = ["人工", "智能", "应用"]
bleu2 = sentence_bleu(
    [reference], prediction, weights=(0.5, 0.5),
    smoothing_function=SmoothingFunction().method1
)

# ROUGE-L 基于最长公共子序列（LCS），不能用 set 交集替代
def lcs_length(a, b):
    dp = [[0] * (len(b)+1) for _ in range(len(a)+1)]
    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            dp[i][j] = dp[i-1][j-1]+1 if a[i-1]==b[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp[-1][-1]

overlap = lcs_length(reference, prediction)
precision = overlap / len(prediction)
recall = overlap / len(reference)
rouge_l = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
print("BLEU-2:", round(bleu2, 4), "ROUGE-L F1:", round(rouge_l, 4))

## 三、逐句解释

1. nn.Transformer(d_model=8,nhead=2) 使用随机输入，仅演示前向传播，输出 (2,2,8)。

2. nltk.sentence_bleu 以参考列表、预测列表和权重 (0.5,0.5) 计算 BLEU-2。

3. lcs_length 使用动态规划找到公共子序列，顺序不同会改变结果。

4. 使用 LCS 长度计算 ROUGE-L precision、recall 与 F1。


## 四、重点函数与参数

### sentence_bleu

- **写法**：`sentence_bleu([reference_tokens], hypothesis_tokens, weights=(0.5,0.5))`
- **参数**：reference_tokens：参考词序列列表；hypothesis_tokens：预测词序列
- **返回**：返回 BLEU 分数
- **特别注意**：短句常用 nltk.translate.bleu_score.SmoothingFunction

### max length of common subsequence

- **写法**：`LCS(reference, prediction)`
- **参数**：两个有顺序的词列表
- **返回**：最长公共子序列长度
- **特别注意**：ROUGE-L 不能简单地用 set 求交集代替

## 五、运行结果核验

Transformer 输出 (2,2,8)；BLEU-2 为 0～1 的数；ROUGE-L F1 约 0.6667。这不是训练完成的翻译模型。

> 运行结果与模型随机性有关，请重点核验形状、指标和文件。


## 六、动手实践（先自己写）

**一级：入门**：手算示例的精确率与召回率。

**二级：巩固**：安装 nltk 后用 sentence_bleu 计算 BLEU-2。

**三级：挑战**：实现最长公共子序列 LCS，再计算简化 ROUGE-L F1。

**本课小挑战**：交换 prediction 中的词语顺序，比较 BLEU-2 与 ROUGE-L F1；解释为什么不能使用 set 交集替代。

建议在下面的单元格中独立填写代码，先从一级入门开始，再做二级、三级。

In [ ]:
# TODO 1：完成一级入门练习

# 在这里输入代码


In [ ]:
# TODO 2：完成二级巩固练习

# 在这里输入代码


In [ ]:
# TODO 3：完成三级挑战练习

# 在这里输入代码


## 七、独立练习自动检查

把本课 BLEU-2 和 ROUGE-L F1 两个数放入 answer_d6 元组。

独立编写 `answer_d6`，再运行检查单元。检查单元不包含答案；失败将显示提示，通过后保存本地结果。


In [ ]:
# TODO 4：完成 answer_d6 后运行
from pathlib import Path as _Path
import sys as _sys
for _candidate in [_Path.cwd(), *_Path.cwd().parents]:
    if (_candidate / "beginner_checkpoints.py").is_file():
        _sys.path.insert(0,str(_candidate))
        break
else:
    raise FileNotFoundError("未找到 beginner_checkpoints.py，请从项目根目录启动 Jupyter")
from beginner_checkpoints import check_lesson
check_lesson("D6",globals())


## 八、常见错误和排查

1. 把随机 Transformer 的输出当作有效翻译结果。

2. 把无序集合交集当作 ROUGE-L，会失去词语顺序信息。

**完成标准**：能够独立解释并运行本课完整示例。


## 九、参考写法（先完成练习再展开）

<details><summary>展开查看原练习参考代码</summary>

```python
print(len(set(reference).intersection(prediction)))
```

参考写法不一定覆盖全部三级拓展练习；应以实际运行结果为准。

</details>

## 十、概念自测

**问题**：BLEU 主要用于评价什么？

A. 图像尺寸

B. 翻译生成质量

C. GPU 温度

<details><summary>展开答案与解析</summary>

正确选项：**B**。BLEU 是翻译任务常用评价指标。

</details>

## 十一、衔接竞赛实操

学完本模块基础课后，返回学习平台，进入 **6.4.4** 竞赛强化题。

[返回零基础课程网页](../../beginner_course.html#D6)

[打开 6.4.4 竞赛实操题](../../practice.html?question=6.4.4)